<p style="font-family: 'Courier New', Courier, monospace; font-size: 40px; font-weight: bold; color: blue;  text-align: center;">
  LoRaWAN Path Loss Measurements in an Indoor Setting: TRANSMIT POWER (TTN ADR REPLAY)
</p>

###  Notebook Summary
The end devices leave LoRaWAN's ADR on, force only the data rate before each uplink and never set a transmit power, so The Things Network chose the power of every uplink through LinkADRReq commands, and the TTN export does not record it. This Notebook replays The Things Stack's ADR (lorawan-stack commit e27dcfc90a, the code on TTN for the whole campaign) on the whole raw export, uplink by uplink in each device session, and writes the transmit power of every uplink to `tx_power_per_uplink.csv`, keyed like the export. Notebook 03 adds it to every row as `tx_power`.

- **Network server:** after each uplink, the margin is the best SNR, over all gateways, of the trailing uplinks at the current data rate since the last accepted LinkADRAns (within the last 20 uplinks), minus the demodulation floor of the SF, minus 15 dB (2.5 dB more while that history holds fewer than 20 uplinks). The sketch changes the data rate after every uplink, so the history is usually one uplink. The server raises the data rate as far as the margin allows, spends the rest in 2 dB power steps, and sends a LinkADRReq when the result differs from the current setting.
- **Device:** it applies every LinkADRReq it receives (the sketch re-forces only the data rate, so the power persists) and answers in its next frame; a join resets it to full power. EU868: at power index k the module's firmware (arduino/mkrwan1300-fw) asks floor(16 − 2k − 2.15) = 13 − 2k dBm and its board driver emits 14 − 2k dBm nominal; the data confirm the 2 dB steps, and each device's real output against the nominal cannot be tested from received data.
- **Export conventions:** an empty frame counter is frame 0, and an empty SNR of a listed gateway is 0 dB. Assumed: every downlink reached the device and every request was accepted.
- **Checks that did not enter the replay:** the SF11 airtime, which grows when a frame carries a LinkADRAns, agrees with the replay on 99.77 % of 284,540 SF11 uplinks; TTN's own packet error rate is reproduced on 99.99 % of uplinks; kerlink001's RSSI steps by +2.02 and −1.88 dB at the rule's two SNR thresholds (±2 dB predicted); an independent implementation gives the same power for all 2,799,900 uplinks both label.

###  1. Libraries and The Things Stack's ADR Constants

In [1]:
# Libraries/Packages Used:
import numpy as np    # For numerical operations (float32 arithmetic, as in The Things Stack).
import pandas as pd   # For data manipulation and analysis.

# The Things Stack's ADR constants, EU868, 125 kHz
F32 = np.float32
FLOOR = {7: F32(-7.5), 8: F32(-10), 9: F32(-12.5), 10: F32(-15), 11: F32(-17.5), 12: F32(-20)}  # demodulation floor per SF (dB)
TX_OFFSET = [F32(-2 * i) for i in range(8)]  # power index 0..7 = 0 to -14 dB below full power
MAX_TP, MAX_ADR_DR = 7, 5
ADR_MARGIN, SAFETY, DR_STEP = F32(15), F32(2.5), F32(2.5)
OPTIMAL, WINDOW = 20, 20

###  2. The Things Stack's ADR After Each Uplink

`adapt` is AdaptDataRate of `adr.go`: the desired data rate, power index and number of transmissions after an uplink, from the ADR uplinks since the last accepted LinkADRAns; `loss_rate` is its frame-loss rate.

In [2]:
def loss_rate(fcnts):
    """adrLossRate: lost frames over the FCnt span of the ADR uplinks (restarting after a counter reset)."""
    if len(fcnts) < 2:
        return 0.0
    lo = last = fcnts[0]
    lost = 0
    for i, f in enumerate(fcnts[1:]):
        if f < last:
            return loss_rate(fcnts[1 + i:])
        if f >= last + 1:
            lost += f - last - 1
        last = f
    return float(F32(lost) / F32(1 + last - lo))


def adapt(recent, cur_dr, cur_tp, cur_nb, last_change):
    """AdaptDataRate after the uplink recent[-1]: the desired (data rate, TX power index, NbTrans)."""
    hist = []
    for dr, f, s in reversed(recent):
        if dr != cur_dr or (last_change != 0 and f < last_change):
            break
        hist.append((dr, f, s))
    hist.reverse()
    snrs = [s for _, _, s in hist if not np.isnan(s)]
    if not snrs:
        return cur_dr, cur_tp, cur_nb
    sf_last = 12 - hist[-1][0]
    margin = F32(max(F32(x) for x in snrs)) - FLOOR[sf_last] - ADR_MARGIN
    optimal = len(hist) >= OPTIMAL
    if not optimal:
        margin = F32(margin - SAFETY)
    # adrAdaptDataRate (min DR index 0, max 5, every DR allowed, min TX power index 0)
    d_dr, d_tp = cur_dr, cur_tp
    max_dr = MAX_ADR_DR
    max_steps = F32(max_dr - d_dr)
    steps = F32((margin - F32(0)) / DR_STEP)
    skip = False
    if F32(0) < steps < max_steps:
        max_dr = d_dr + int(steps)
    elif steps <= 0:
        skip = True
    if not skip and max_dr > d_dr:
        margin = F32(margin - F32(max_dr - d_dr) * DR_STEP)
        d_dr, d_tp = max_dr, 0
    # adrAdaptTxPowerIndex (min 0, max 7, nothing rejected)
    for idx in range(MAX_TP, -1, -1):
        diff = F32(TX_OFFSET[d_tp] - TX_OFFSET[idx])
        if diff > margin and idx != 0:
            continue
        if not optimal and diff < 0 and -diff <= SAFETY:
            break
        margin = F32(margin - diff)
        d_tp = idx
        break
    # adrAdaptNbTrans (no NbTrans settings: the clamp is the identity)
    nb = cur_nb
    if len(hist) >= OPTIMAL // 2:
        lr = loss_rate([f for _, f, _ in hist])
        if lr < 0.05:
            nb = 1 + nb // 3
        elif lr < 0.10:
            pass
        elif lr < 0.30:
            nb = 2 + nb // 2
        else:
            nb = 3
    return d_dr, d_tp, nb

###  3. One Device Session, Uplink by Uplink

The server's MAC state and the device's power index, uplink by uplink (`grpc_gsns.go` and `link_adr.go`): a request computed after one uplink reaches the device on the downlink, and its answer rides in the next frame.

In [3]:
def replay_session(dr, f_cnt, max_snr):
    """TX power index of each uplink of one session, given its uplinks in frame-counter order."""
    cur_dr, cur_tp, cur_nb, last_change = 0, 0, 1, 0  # the server's MAC state after the join
    dev_tp = 0                                        # the device's power index after the join
    recent, pending, f_prev = [], None, None
    index = np.zeros(len(dr), dtype=np.int16)
    for i in range(len(dr)):
        d, f, s = int(dr[i]), int(f_cnt[i]), float(max_snr[i])
        if pending is not None:
            dev_tp = pending[1]                       # the device applied the request on the downlink
            if f == f_prev + 1:                       # its answer rides in the next frame, received here
                cur_dr, cur_tp = pending[0], pending[1]
                if pending[2] > 0 and cur_nb != pending[2]:
                    cur_nb = pending[2]
                last_change = f
        index[i] = dev_tp
        recent.append((d, f, s))
        if len(recent) > WINDOW:
            recent = recent[-WINDOW:]
        if d < cur_dr:
            cur_tp = 0
        cur_dr = d
        desired = adapt(recent, cur_dr, cur_tp, cur_nb, last_change)
        pending = desired if desired != (cur_dr, cur_tp, cur_nb) else None
        f_prev = f
    return index

###  4. Load the Raw Export

All dates and all gateways: the ADR state carries over from one day to the next, and it uses the SNR of every gateway that received the uplink. The same uplink ingested twice has the same device, session, frame counter and reception time.

In [4]:
P = 'uplink_message_'
key = ['end_device_ids_device_id', f'{P}session_key_id', f'{P}f_cnt']
rx = [f'{P}rx_metadata_{k}_{f}' for k in range(4) for f in ('snr', 'gateway_ids_gateway_id')]
raw = pd.read_csv('../all_data_files/unsorted_combined_measurements_data.csv', low_memory=False,
                  usecols=key + [f'{P}received_at', f'{P}settings_data_rate_lora_spreading_factor'] + rx)

# One row per uplink
up = raw.drop_duplicates(key + [f'{P}received_at']).copy()
assert not up.duplicated(key).any()
up['f'] = up[f'{P}f_cnt'].fillna(0).astype(np.int64)  # frame 0 (the first after a join) is exported empty
up['dr'] = (12 - up[f'{P}settings_data_rate_lora_spreading_factor']).astype(int)
# the best SNR over the gateways that received the uplink; TTS omits an SNR of exactly 0 from the export
snr = [up[f'{P}rx_metadata_{k}_snr'].where(up[f'{P}rx_metadata_{k}_gateway_ids_gateway_id'].isna(), up[f'{P}rx_metadata_{k}_snr'].fillna(0.0))
       for k in range(4)]
up['max_snr'] = pd.concat(snr, axis=1).max(axis=1)
# sessions in the order they began, uplinks in frame-counter order
up['t'] = pd.to_datetime(up[f'{P}received_at'], utc=True, format='ISO8601')
up['first'] = up.groupby(key[:2])['t'].transform('min')
up = up.sort_values([key[0], 'first', 'f'], kind='mergesort')
print(f"{len(raw):,} rows, {len(up):,} uplinks, {up.groupby(key[:2]).ngroups} sessions")

3,377,347 rows, 2,799,925 uplinks, 38 sessions


###  5. Replay Every Session

In [5]:
index = [pd.Series(replay_session(g['dr'].to_numpy(), g['f'].to_numpy(), g['max_snr'].to_numpy()), index=g.index)
         for _, g in up.groupby(key[:2], sort=False)]
up['tx_power'] = 14 - 2 * pd.concat(index)  # dBm

print("Uplinks by transmit power (dBm):", up['tx_power'].value_counts().sort_index().to_dict())
print((100 * (up['tx_power'] < 14)).groupby(up[key[0]]).mean().round(1).rename('uplinks below 14 dBm (%)').to_string())

Uplinks by transmit power (dBm): {0: 1, 2: 3, 4: 8, 6: 16, 8: 334, 10: 17916, 12: 913814, 14: 1867833}
end_device_ids_device_id
pilotdevice      36.9
pilotdevice01    33.5
pilotdevice02    34.4
pilotdevice03    39.0
pilotdevice04    29.9
pilotdevice05    26.1
pilotdevice06     8.3


###  6. Save the Transmit Power of Every Uplink

One row per uplink, with the export's own key columns (an empty frame counter stays empty), so that every row of the export finds its power.

In [6]:
up.sort_values([key[0], 't'])[key + [f'{P}received_at', 'tx_power']].to_csv('../all_data_files/tx_power_per_uplink.csv', index=False)
print("Saved ../all_data_files/tx_power_per_uplink.csv")

Saved ../all_data_files/tx_power_per_uplink.csv
